In [7]:
from tqdm.auto import tqdm
import numpy as np

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [1]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data

documents = load_faq_data()

## 03-Embeddings Dataset

In [2]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [6]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1400

In [8]:
X = np.array(vectors)

In [9]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [11]:
scores = X.dot(v_query)

In [20]:
idx = np.argmax(scores)
# idx, scores[idx]
# documents[idx]

top5 = np.argsort(-scores)[:5]
print(scores[top5])

for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

[0.762941   0.7579372  0.7192131  0.6536311  0.56009984]
0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machin

## 05- Min Search

In [21]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [22]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [23]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [26]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
results[1]

{'id': '69d122f12e',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'}